# Prompting Strategies (Module 3 · Applications of GenAI)

Traditional-half add-on for **Session 03**. Hands-on tour of zero-/few-shot, chain-of-thought, and structured prompting with Hugging Face `transformers`. Pairs with the Agentic-AI half (Lecture 1: ReAct) where these prompts become the agent's reasoning loop.


In [ ]:
# Install dependencies (run once). On Colab these are usually present, but this
# guarantees a compatible, up-to-date setup:
# !pip install -q -U transformers torch sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Flan-T5 is a small, free, *instruction-tuned* model. Unlike GPT-2 (a raw text
# completer from 2019), it actually follows instructions like "Classify..." or
# "Explain...", so these prompting demos produce sensible answers. Downloads once (~1 GB).
#
# IMPORTANT: we load the model + tokenizer directly instead of using
#   pipeline("text2text-generation", ...)
# The "text2text-generation" pipeline task was REMOVED in transformers v5, so the
# old code fails on Colab with:
#   KeyError: "Unknown task text2text-generation ..."
# Loading the model ourselves works on every transformers version.
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

def ask(prompt, max_new_tokens=60):
    # Greedy decoding by default -> clean, reproducible answers for demos.
    # (T5 returns only the answer, not prompt + continuation, so no slicing needed.)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    print(tokenizer.decode(output_ids[0], skip_special_tokens=True).strip())


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
# ===============================
# 1. Zero-shot Prompting
# ===============================
print("---- ZERO SHOT ----")
ask("Classify the sentiment of this sentence: 'I really enjoyed the concert.'")


In [ ]:
# ===============================
# 2. One-shot Prompting
# ===============================
print("---- ONE SHOT ----")
prompt = """Classify the sentiment of these sentences.

Sentence: The food was terrible.
Sentiment: Negative

Sentence: I really enjoyed the concert.
Sentiment:"""
ask(prompt)

In [ ]:
# ===============================
# 3. Few-shot Prompting
# ===============================
prompt = """Classify the sentiment of these sentences.

Sentence: I love pizza.
Sentiment: Positive

Sentence: This is the worst movie ever.
Sentiment: Negative

Sentence: I love dancing.
Sentiment:"""
ask(prompt)

In [ ]:
# ===============================
# 3b. Few-shot Prompting (more examples)
# ===============================
print("---- FEW SHOT (more examples) ----")
prompt = """Classify the sentiment of these sentences.

Sentence: I love pizza.
Sentiment: Positive

Sentence: This is the worst movie ever.
Sentiment: Negative

Sentence: The weather is nice today.
Sentiment: Positive

Sentence: I feel sick today.
Sentiment: Negative

Sentence: That ice cream was so good!
Sentiment: Positive

Sentence: I like music
Sentiment: Positive

Sentence: I like dancing
Sentiment:"""
ask(prompt)


In [ ]:
# ===============================
# 4. Chain-of-Thought Prompting
# (Model is guided to reason step by step)
# ===============================
print("---- CHAIN OF THOUGHT ----")
prompt = """Question: If there are 3 apples and you take away 1, how many are left?
Let's think step by step."""
ask(prompt)



In [ ]:
# ===============================
# 5. Role Prompting (persona: tutor)
# ===============================
print("---- ROLE PROMPT: math tutor ----")
prompt = """You are a helpful math tutor.
Explain Pythagoras' theorem to a 10-year-old in simple words."""
ask(prompt, max_new_tokens=80)

# ===============================
# 6. Role Prompting (persona: pirate)
# ===============================
print("\n---- ROLE PROMPT: pirate ----")
prompt = """You are a pirate. Answer in pirate language.
Question: What is your favorite food?"""
ask(prompt, max_new_tokens=80)


In [ ]:
# ===============================
# 📝 Student Activity
# ===============================
# Try these on your own:
# 1. Write a ZERO-SHOT prompt to identify if a number is even or odd.
# 2. Create a ONE-SHOT example for topic classification (sports, politics, tech).
# 3. Write a FEW-SHOT prompt for extracting names from sentences.
# 4. Use CHAIN-OF-THOUGHT to solve: "If a train travels 60km in 1 hour, how far in 3 hours?"
# 5. ROLE PROMPT: Ask the model to act as a teacher and explain the moon to a 7-year-old.

your_prompt = """<write your own prompt here>"""
ask(your_prompt)